# Assignment 03 — development error review and final freeze

## tl;dr

All 4,203 development feature records and all six fitted-model hashes reproduce the original experiment exactly. RF hybrid F1 drops 7.21 percentage points under complexity-block shuffle and 5.72 points within password/language/shard cells. The password-present F1 gap has an exploratory group-bootstrap interval of −15.53 to +1.69 points. Keep the frozen internal benchmark with its limitations; test stays sealed.

## Context & Methods

The review reproduces the exact 4,203 development feature rows and six baseline models. Complexity bins use train tag-count quartiles. Password gaps use whole-group bootstrap; standardization is descriptive. Permutation means measure input reliance under artificial feature shuffles, not adversarial robustness.

### Key Assumptions

Published source labels are not independently adjudicated. All shards belong to one historical source cohort. The 30 shuffle repeats quantify shuffle variability; they are not a population confidence interval. No feature, threshold, model or seed is selected from these diagnostics.

## Data

Run from the repository or its notebooks directory. Only committed sanitized evidence is read; no live page, raw HTML or test feature matrix is accessed.

In [ ]:
from pathlib import Path
import json, hashlib, sys
root = Path.cwd()
if not (root / 'config').exists():
    root = root.parent
assert (root / 'config/assignment03_final_evaluation_v1.json').exists()
sys.path.insert(0, str(root / 'src'))
from phishing_url.final_evaluation_contract import verify
freeze_check = verify(root)
print(freeze_check)
evidence = root / 'results/assignment03/review_v1'
load = lambda name: json.loads((evidence / name).read_text())
summary = load('review_summary.json')
assert summary['test_feature_rows'] == summary['test_predictions'] == 0
print({k: summary[k] for k in ['replayed_fits', 'maximum_score_difference', 'feature_hashes_match_original', 'validation_rows', 'validation_groups']})

## Results

### Password-present and password-absent pages

In [ ]:
condition = 'random_forest/url_dom/baseline'
for row in load('slices.json'):
    if row['condition'] == condition and row['facet'] == 'password_present':
        print({k: row[k] for k in ['value','n','positive_n','negative_n','fp','fn','f1','recall','fpr']})
print('Present minus absent, exploratory group-bootstrap intervals:')
print(load('password_gap_intervals.json')[condition])
print('Complexity-standardized rates and retained support:')
print(load('password_complexity_standardization.json')[condition])

### Complexity and feature reliance

In [ ]:
print(load('complexity_distributions.json'))
for row in load('permutation_summary.json'):
    if row['condition'] == condition:
        print(row['mode'], row['block'], 'mean F1 loss (percentage points):', round(100 * row['f1_drop']['mean'], 3), 'shuffle SD:', round(100 * row['f1_drop']['std_sample'], 3), 'exchangeable rows:', row['eligible_rows'])

### Hybrid changes relative to each comparator

In [ ]:
for row in load('paired_error_cohorts.json'):
    if row['model'] == 'random_forest' and row['kind'] in ['fixes', 'breaks']:
        print(row)
assert load('independent_verification.json')['verification_passed']
print('All reference comparisons and independent evidence checks passed.')

## Takeaways

The simplest validation pages contain 30 benign / 153 phishing examples; the most complex contain 142 / 32. RF hybrid FPR is 26.67% in the simplest bin; phishing recall is 75.00% in the most complex. Password-group error-rate gaps shrink after descriptive complexity standardization. These are cohort associations, not causal or adversarial evidence.

No correctness blocker was found. The six original cells, seed and threshold stay unchanged. The frozen specification is checked without test inference; one-time final test execution and the submission report are subsequent work.